In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import folium
from folium import plugins

subway = pd.read_csv('/content/drive/MyDrive/데이터분석프로젝트/강서구 공모전/데이터/서울시 역사마스터 정보.csv', encoding='CP949')

In [ ]:
subway.head(10)

,역사_ID,역사명,호선,위도,경도
0,9996,미사,5호선,127.193877,37.560927
1,9995,강일,5호선,127.175930,37.557490
2,4929,김포공항,김포골드라인,126.801868,37.562360
3,4928,고촌,김포골드라인,126.770345,37.601243
4,4927,풍무,김포골드라인,126.732387,37.612488
5,4926,사우(김포시청),김포골드라인,126.719731,37.620249
6,4925,걸포북변,김포골드라인,126.705975,37.631650
7,4924,운양,김포골드라인,126.683930,37.653867
8,4923,장기,김포골드라인,126.669017,37.643986
9,4922,마산,김포골드라인,126.644344,37.640732


In [ ]:
# 위도, 경도 컬럼이 이상하게 적혀있으므로 수정

subway = subway.rename(columns={'위도':'경도','경도':'위도'})

In [ ]:
subway.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   역사_ID   765 non-null    int64  
 1   역사명     765 non-null    object 
 2   호선      765 non-null    object 
 3   경도      765 non-null    float64
 4   위도      765 non-null    float64
dtypes: float64(2), int64(1), object(2)
memory usage: 30.0+ KB


### 분석에서 활용할 수 있는 정보
##### 역사_ID, 역사명, 호선, 위도, 경도
- 역사_ID는 역사명과 호선에 따라 부여된 고유값.
- 지하철역의 역사명과 호선을 알 수 있다,
- 위도, 경도를 통해 강서구 내 역사의 분포를 확인 가능.

In [ ]:
# 호선마다 역사_ID가 다르므로 역사명이 중복되는 경우가 있음.
# 강서구 내 모든 역이 5호선, 9호선을 지나므로 호선이 5,9호선인 행들을 제외하고 제거

display(subway[subway['역사명']=='마곡나루'])
display(subway[subway['역사명']=='김포공항'])

subway_gs = subway[(subway['역사명']=='방화') | 
                   (subway['역사명']=='개화') |
                   (subway['역사명']=='개화산') |
                   (subway['역사명']=='김포공항') |
                   (subway['역사명']=='공창시장') |
                   (subway['역사명']=='신방화') |
                   (subway['역사명']=='마곡나루') |
                   (subway['역사명']=='송정') |
                   (subway['역사명']=='마곡') |
                   (subway['역사명']=='발산') |
                   (subway['역사명']=='우장산') |
                   (subway['역사명']=='화곡') |
                   (subway['역사명']=='까치산') |
                   (subway['역사명']=='양천향교') |
                   (subway['역사명']=='가양') |
                   (subway['역사명']=='증미') |
                   (subway['역사명']=='등촌') |
                   (subway['역사명']=='염창')
                  ]

,역사_ID,역사명,호선,경도,위도
103,4206,마곡나루,공항철도1호선,126.827378,37.565543
141,4105,마곡나루,9호선,126.827310,37.566778


,역사_ID,역사명,호선,경도,위도
2,4929,김포공항,김포골드라인,126.801868,37.562360
102,4207,김포공항,공항철도1호선,126.801904,37.561842
144,4102,김포공항,9호선,126.802152,37.561916
375,2513,김포공항,5호선,126.801292,37.562384


In [ ]:
# 역사명이 겹치는 곳들 제거

subway_gs.drop(index=[2,102,103,144], inplace=True)

C:\Users\chlwj\AppData\Local\Temp\ipykernel_19300\3265844110.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subway_gs.drop(index=[2,102,103,144], inplace=True)


In [ ]:
subway_gs.reset_index(drop=True, inplace=True)

In [ ]:
# 결측치인 값 없는 것을 확인

subway_gs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   역사_ID   17 non-null     int64  
 1   역사명     17 non-null     object 
 2   호선      17 non-null     object 
 3   경도      17 non-null     float64
 4   위도      17 non-null     float64
dtypes: float64(2), int64(1), object(2)
memory usage: 808.0+ bytes


In [ ]:
subway_gs

,역사_ID,역사명,호선,경도,위도
0,4110,염창,9호선,126.874916,37.546936
1,4109,등촌,9호선,126.865689,37.550632
2,4108,증미,9호선,126.861939,37.557402
3,4107,가양,9호선,126.854456,37.561391
4,4106,양천향교,9호선,126.841333,37.568381
5,4105,마곡나루,9호선,126.827310,37.566778
6,4104,신방화,9호선,126.816601,37.567532
7,4101,개화,9호선,126.798153,37.578608
8,2519,까치산,5호선,126.846683,37.531768
9,2518,화곡,5호선,126.840461,37.541513


In [ ]:
location_data = subway_gs[['위도','경도']].values[:len(subway_gs)].tolist()
center =[subway_gs.위도.mean(),subway_gs.경도.mean()]

In [ ]:
map = folium.Map(location=center,zoom_start=13,width='70%')
for i in range(len(subway_gs)):
    subway_name=subway_gs.역사명[i]
    station_color='black'
    folium.Circle(location=location_data[i],radius=1,color=station_color,fill=True,fill_opacity=1, tooltip=subway_name).add_to(map)

plugins.HeatMap(location_data,radius=20).add_to(map)

In [ ]:
map

#### 강서구청 부근을 제외하고는 지하철 역이 골고루 분포된 모습.